<a href="https://colab.research.google.com/github/anjalidrj/GO-analysis-of-peptides/blob/main/20260701_cpep_genome_scan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install biopython pandas tqdm

In [ ]:
!pip install pyahocorasick -q

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from Bio import SeqIO

# Check rice transcriptome
grape = list(SeqIO.parse("sol.fasta", "fasta")) #change fasta file name
print(f"grape transcripts: {len(grape)}")

# Check peptides
peptides = list(SeqIO.parse("peptides_mc0_A.fasta", "fasta")) #change fasta file name
print(f"Peptides: {len(peptides)}")
print("\nYour peptides:")
for p in peptides:
    print(f"  {p.id}: {p.seq}")

In [ ]:
print(f"Total peptides: {len(peptides)}")

# check hydrophilicity of each
HYDROPHILIC = set("DNEQKR")

passing = []
failing = []

for p in peptides:
    seq = str(p.seq)
    score = sum(1 for aa in seq if aa in HYDROPHILIC) / len(seq)
    passes = "✓" if score > 0.33 else "✗"
    if score > 0.33:
        passing.append(p.id)
    else:
        failing.append(p.id)
    print(f"{passes} {p.id}: {seq} ({score:.0%} hydrophilic)")

print(f"\nPassing hydrophilicity filter: {len(passing)}/{len(peptides)}")
print(f"Failing hydrophilicity filter: {len(failing)}/{len(peptides)}")

In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq
from tqdm import tqdm

CHUNK_SIZE = 1000  # process 1000 transcripts at a time

print(f"Total rice transcripts: {len(grape)}")
print(f"Processing in chunks of {CHUNK_SIZE}...")

with open("cotton_3frames.fasta", "w") as out:
    for i in tqdm(range(0, len(grape), CHUNK_SIZE)):
        chunk = grape[i:i + CHUNK_SIZE]
        for record in chunk:
            seq = record.seq
            for frame in range(3):
                trimmed = seq[frame:]
                trimmed = trimmed[:len(trimmed) - len(trimmed) % 3]
                if len(trimmed) < 3:
                    continue
                protein = trimmed.translate()
                out.write(f">{record.id}_frame{frame + 1}\n{protein}\n")

print("Done! Verifying output...")
frames = list(SeqIO.parse("cotton_3frames.fasta", "fasta"))
print(f"Total translated frame sequences: {len(frames)}")

In [ ]:
import ahocorasick
from tqdm import tqdm
import pandas as pd

HYDROPHILIC = set("DNEQKR")

# get only passing peptides
passing_peptides = [p for p in peptides if
                    sum(1 for aa in str(p.seq) if aa in HYDROPHILIC) / len(str(p.seq)) > 0.33]

print(f"Scanning {len(passing_peptides)} peptides against {len(frames)} translated frames...")

# Build one combined search automaton out of all peptides (do this once)
seq_to_peptide_ids = {}
for pep in passing_peptides:
    pep_seq = str(pep.seq).upper()
    seq_to_peptide_ids.setdefault(pep_seq, []).append(pep.id)

automaton = ahocorasick.Automaton()
for pep_seq in seq_to_peptide_ids:
    automaton.add_word(pep_seq, pep_seq)
automaton.make_automaton()

hits = []
for frame_record in tqdm(frames):
    target_seq = str(frame_record.seq).upper()
    frame_num = frame_record.id.split("_frame")[-1]
    transcript_id = frame_record.id.rsplit("_frame", 1)[0]

    seen_this_frame = set()  # keep first-occurrence-only behavior, matching your original script
    for end_index, pep_seq in automaton.iter(target_seq):
        if pep_seq in seen_this_frame:
            continue
        seen_this_frame.add(pep_seq)
        start_index = end_index - len(pep_seq) + 1
        for pep_id in seq_to_peptide_ids[pep_seq]:
            hits.append({
                "peptide_id": pep_id,
                "peptide_seq": pep_seq,
                "transcript_id": transcript_id,
                "frame": f"+{frame_num}",
                "position_aa": start_index,
            })

print(f"\nTotal hits: {len(hits)}")
df = pd.DataFrame(hits)
df.to_csv("exact_hits.csv", index=False)
print("Saved to exact_hits.csv")

In [ ]:
print(f"Total hits: {len(df)}")
print(f"Unique peptides with at least one hit: {df['peptide_id'].nunique()}")
print(f"Unique transcripts hit: {df['transcript_id'].nunique()}")
print(f"\nHits per frame:")
print(df['frame'].value_counts())
print(f"\nTop 20 most hit transcripts:")
print(df['transcript_id'].value_counts().head(20))
print(f"\nPeptides with most hits:")
print(df['peptide_id'].value_counts().head(20))

In [ ]:
# add hydrophilicity score to the hits dataframe
def hydrophilicity(seq):
    return sum(1 for aa in seq if aa in HYDROPHILIC) / len(seq)

df["hydrophilicity"] = df["peptide_seq"].apply(hydrophilicity)
df["peptide_length"] = df["peptide_seq"].apply(len)

# sort by transcript hits (fewer = more specific)
specificity = df.groupby("peptide_id").size().rename("total_transcript_hits")
df = df.merge(specificity, on="peptide_id")
df = df.sort_values("total_transcript_hits")

In [ ]:
# build a lookup dictionary: transcript_id -> gene description
id_to_description = {}
for record in grape:
    # remove the accession from the start of the description to get just the gene name
    desc = record.description.replace(record.id, "").strip()
    id_to_description[record.id] = desc

# verify it works
print(list(id_to_description.items())[:5])

In [ ]:
# add gene description to hits dataframe
df["gene_description"] = df["transcript_id"].map(id_to_description)

# reorder columns nicely
df = df[[
    "peptide_id",
    "peptide_seq",
    "peptide_length",
    "hydrophilicity",
    "transcript_id",
    "gene_description",
    "frame",
    "position_aa",
    "total_transcript_hits"
]]

# sort by specificity (fewer hits = more specific)
df = df.sort_values("total_transcript_hits")

# --- NEW: pull GLYMA_ gene id out of the gene_description column ---
gdcol = "gene_description" if "gene_description" in df.columns else df.columns[5]
df["gene_id"] = df[gdcol].str.extract(r"gene:(\S+)")

df.to_csv("exact_hits_final_sol.csv", index=False)
print(f"Total hits: {len(df)}")
print(f"genes extracted: {df['gene_id'].notna().sum()} / {len(df)}")
print(f"\nSample output:")
print(df.head(10).to_string())

# download
from google.colab import files
files.download("exact_hits_final_sol.csv")

In [ ]:
# add gene description to hits dataframe
df["gene_description"] = df["transcript_id"].map(id_to_description)

# reorder columns nicely
df = df[[
    "peptide_id",
    "peptide_seq",
    "peptide_length",
    "hydrophilicity",
    "transcript_id",
    "gene_description",
    "frame",
    "position_aa",
    "total_transcript_hits"
]]

# sort by specificity (fewer hits = more specific)
df = df.sort_values("total_transcript_hits")

# --- get clean gene_id: strip the 'gene-' prefix, keep the version (.3 etc.) ---
df["gene_id"] = df["transcript_id"].str.replace(r"^gene-", "", regex=True)

df.to_csv("exact_hits_final_sol.csv", index=False)
print(f"Total hits: {len(df)}")
print(f"genes extracted: {df['gene_id'].notna().sum()} / {len(df)}")
print("\nSample gene_id values:")
print(df["gene_id"].head(10).to_string(index=False))

# download
from google.colab import files
files.download("exact_hits_final_sol.csv")

In [ ]:
from Bio import SeqIO
from google.colab import files

# split into 3 frame files
for frame_num in [1, 2, 3]:
    filename = f"tomato_frame{frame_num}.fasta"
    print(f"Writing {filename}...")

    with open(filename, "w") as out:
        for record in frames:
            if record.id.endswith(f"_frame{frame_num}"):
                out.write(f">{record.id}\n{record.seq}\n")

    count = sum(1 for r in SeqIO.parse(filename, "fasta"))
    print(f"  {count} sequences written")

print("\nDownloading...")
files.download("tomato_frame1.fasta")
files.download("tomato_frame2.fasta")
files.download("tomato_frame3.fasta")